It's a bit messy since I had to download data on Google Colab. I started the project while on summer vacation back home in Dubai, where Polymarket is banned, so this is the only way to circumvent the data restrictions. 

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

import os
os.makedirs('/content/drive/MyDrive/polymarket', exist_ok=True)

In [ ]:
%cd /content/drive/MyDrive/polymarket

In [ ]:
# runs the actual download script from claude and backs it up on google drive 
# just in case colab's runtime crashes
!python download_price_history.py \
    --input finalsample.parquet \
    --db /content/price_history.db \
    --backup-path /content/drive/MyDrive/polymarket/price_history_backup.db \
    --backup-interval 1800 \
    --fidelity 60 --chunk-days 15 --concurrency 40 --rate-limit 80 \
    --stall-timeout 90 \
    --both-outcomes

In [ ]:
!python export_to_parquet.py --db /content/price_history.db --out /content/price_history.parquet

In [ ]:
from google.colab import files
files.download('/content/price_history.parquet')

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('/content/drive/MyDrive/polymarket/recovered.db')

# see the actual price data it pulled
df = pd.read_sql_query("SELECT * FROM price_history LIMIT 20", conn)
df

In [ ]:
import pandas as pd
markets = pd.read_parquet("/content/drive/MyDrive/polymarket/polymarket_closed_markets.parquet")

market_ids = markets["id"].tolist()

In [ ]:
import requests
import pandas as pd
import time

rows = []

for market_id in market_ids:

    r = requests.get(
        f"https://gamma-api.polymarket.com/markets/{market_id}/tags",
        timeout=30,
    )

    r.raise_for_status()

    rows.append({
        "market_id": market_id,
        "tags": r.json()      # list of tag dictionaries
    })

    time.sleep(0.05)   # be polite to the API

tags_df = pd.DataFrame(rows)
tags_df.to_parquet("market_tags.parquet", index=False)